In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon, kruskal
import scikit_posthocs as sp

In [7]:
def convert_2_int(num):
    if pd.isna(num) or num == '-':
        return None
    return int(num)

# Study 1(RAVEN_1): Multi-stage Accuracy and Confidence

In [5]:
# Load the dataset
main_folder = 'experiment_1_RAVEN'
df_raven_1 = pd.read_csv(f'{main_folder}/results/experiment_runs_log_users_actions_changes_only.csv')
df_experiment_details_raven_1 = pd.read_csv(f"{main_folder}/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv", header=0, usecols=range(10))

In [29]:
def analysis_test_results(users, users_logs):
    data = {
      "participants": []
    }

    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        user_details = {"id": user, 
                            "accuracy": {"baseline": 0, "prediction": 0, "explanation": 0},
                            "confidence": {"baseline": 0, "prediction": 0, "explanation": 0},
                            "n_valid_questions": 0,
                           }
        
        for _, row in user_data.iterrows():
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            pred_choice = convert_2_int(row['predictedShapeChoiceIndex'])
            before_ai = convert_2_int(row['lInd_0_BeforeAi'])
            after_ai = convert_2_int(row['lInd_1_AfterAi'])
            after_xai = convert_2_int(row['lInd_2_AfterXai'])
            explanation_type = row['param1']
            confidence_before_ai = convert_2_int(row['face_0_BeforeAi'])
            confidence_after_ai = convert_2_int(row['face_1_AfterAi'])
            confidence_after_xai = convert_2_int(row['face_2_AfterXai'])

            # Filter rows that contain question data
            if explanation_type in ('LLM', 'OS'):

                if pd.notna(before_ai) and pd.notna(after_ai) and pd.notna(after_xai):
                    if before_ai == true_choice:
                        user_details["accuracy"]["baseline"] += 1
                    if after_ai == true_choice:
                        user_details["accuracy"]["prediction"] += 1
                    if after_xai == true_choice:
                        user_details["accuracy"]["explanation"] += 1
                    user_details["confidence"]["baseline"] += confidence_before_ai
                    user_details["confidence"]["prediction"] += confidence_after_ai
                    user_details["confidence"]["explanation"] += confidence_after_xai
                    user_details["n_valid_questions"] += 1

        data["participants"].append(user_details)

    return data

In [30]:
data = analysis_test_results(set(df_raven_1['tuid']), df_raven_1)

#### Extracting arrays for statistics

In [32]:
acc_baseline = []
acc_prediction = []
acc_explanation = []

for p in data["participants"]:
    acc_baseline.append(p["accuracy"]["baseline"])
    acc_prediction.append(p["accuracy"]["prediction"])
    acc_explanation.append(p["accuracy"]["explanation"])

In [33]:
conf_baseline = []
conf_prediction = []
conf_explanation = []

for p in data["participants"]:
    n = p["n_valid_questions"]
    if n == 0:
        continue

    conf_baseline.append(p["confidence"]["baseline"] / n)
    conf_prediction.append(p["confidence"]["prediction"] / n)
    conf_explanation.append(p["confidence"]["explanation"] / n)

#### Tests: Friedman test + Kendall’s W

In [35]:
def friedman_with_kendalls_w(x, y, z):
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)

    chi2, p = friedmanchisquare(x, y, z)

    n = len(x)     
    k = 3       
    W = chi2 / (n * (k - 1))

    return chi2, p, W

In [36]:
chi2_acc, p_acc, W_acc = friedman_with_kendalls_w(
    acc_baseline, acc_prediction, acc_explanation
)

print(f"Accuracy: χ²(df=2)={chi2_acc:.3f}, p={p_acc:.4f}, Kendall’s W={W_acc:.3f}")

chi2_conf, p_conf, W_conf = friedman_with_kendalls_w(
    conf_baseline, conf_prediction, conf_explanation
)

print(f"Confidence: χ²(df=2)={chi2_conf:.3f}, p={p_conf:.4f}, Kendall’s W={W_conf:.3f}")

Accuracy: χ²(df=2)=37.978, p=0.0000, Kendall’s W=0.703
Confidence: χ²(df=2)=7.095, p=0.0288, Kendall’s W=0.131


#### Post-hoc: Wilcoxon (prediction vs explanation)

In [38]:
def wilcoxon_posthoc(x, y):
    x = np.array(x)
    y = np.array(y)

    stat, p = wilcoxon(x, y)
    return stat, p

In [39]:
w_acc, p_acc_post = wilcoxon_posthoc(acc_prediction, acc_explanation)
print(f"Accuracy post-hoc: Wilcoxon W={w_acc:.3f}, p={p_acc_post:.4f}")

w_conf, p_conf_post = wilcoxon_posthoc(conf_prediction, conf_explanation)
print(f"Confidence post-hoc: Wilcoxon W={w_conf:.3f}, p={p_conf_post:.4f}")

Accuracy post-hoc: Wilcoxon W=75.500, p=0.9596
Confidence post-hoc: Wilcoxon W=42.000, p=0.0101


# Study 2(RAVEN_2): Group Accuracy Comparisons

In [8]:
# Load the dataset
main_folder = 'experiment_2_RAVEN'

# Groups a-d
df = pd.read_csv(f'{main_folder}/results/experiment_runs_log_users_actions_changes_only.csv')
df_experiment_details = pd.read_csv(f"{main_folder}/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv", header=0, usecols=range(11))
# Group e
df_group_e = pd.read_csv(f'{main_folder}/results/group_e/experiment_runs_log_users_actions_changes_only.csv')
df_experiment_details_group_e = pd.read_csv(f"{main_folder}/results/group_e/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv", header=0, usecols=range(11))

df_raven_2 = pd.concat([df, df_group_e], ignore_index=True)
df_experiment_details_raven_2 = pd.concat([df_experiment_details, df_experiment_details_group_e], ignore_index=True)

In [9]:
def group_E_add_explanation_feedback(
    tuid,
    faceSelected_tuple,
    question_num_tuple=(1, 2, 3)
):
    rows = []

    for faceSelected, question_num in zip(faceSelected_tuple, question_num_tuple):
        rows.append({
            "tuid": tuid,
            "ugroup": "E",
            "explanationType": np.nan,
            "faceSelected": faceSelected,
            "selectedShapeIndex": np.nan,
            "predictedShapeChoiceIndex": np.nan,
            "trueShapeChoiceIndex": np.nan,
            "isCorrectPrediction": np.nan,
            "stepActionType": np.nan,
            "param1": np.nan,
            "param2": np.nan,
            "param3": np.nan,
            "param4": np.nan,
            "blockId": f"block_end_questionnaire_explanation__question_{question_num}",
            "actionName": "click",
            "objectLogName": f"block_end_questionnaire_explanation__question_{question_num}_SubmitRatingBtn",
            "date": np.nan,
            "time": np.nan,
        })

    return pd.DataFrame(rows)

dfs = []
dfs.append(group_E_add_explanation_feedback("UID_Hg08dP61na51ZM56rQ", (3,4,3)))
dfs.append(group_E_add_explanation_feedback("UID_TB24Xd47rO50uC82bW", (3,5,5)))
dfs.append(group_E_add_explanation_feedback("UID_qC25No67vP11AB03bR", (5,5,5)))
dfs.append(group_E_add_explanation_feedback("UID_Xc25bq55Ic87Qs53bk", (4,4,5)))
dfs.append(group_E_add_explanation_feedback("UID_Iv20VI44US80dS78xY", (5,5,5)))
dfs.append(group_E_add_explanation_feedback("UID_wB23Fp88kp97MH01LP", (5,4,5)))
dfs.append(group_E_add_explanation_feedback("UID_zJ06Xm51mv59uw34ZF", (1,3,5)))
dfs.append(group_E_add_explanation_feedback("UID_Eb25VY43sm07Yf87ww", (5,5,5)))
dfs.append(group_E_add_explanation_feedback("UID_tL21Td42TW96jx50JE", (5,5,4)))
dfs.append(group_E_add_explanation_feedback("UID_vL31FS86ri66rZ14sT", (3,5,5)))
dfs.append(group_E_add_explanation_feedback("UID_EI38ib82HG46fU08oR", (4,5,3)))
dfs.append(group_E_add_explanation_feedback("UID_Rs55MY88wb21UN85TJ", (3,5,3)))

df_raven_2 = pd.concat([df_raven_2, *dfs], ignore_index=True)

## Delete problematic users
bad_tuids = ["UID_Kg24ef54aE73wd38JF", "UID_pk73wE89io45VZ91BF", "UID_JG54wN91du61Ls34Ng", "UID_UA64iV84bd20uJ55aO", "UID_UA64iV84bd20uJ55aO", "UID_TEST_TEST_1",
            "UID_ZV94Gd16sF84aL62eH", "UID_Pv48eG17sn92Ge75sg", "UID_Pv48eG17sn92Ge75sg", "UID_UA64iV84bd20uJ55aO"]
df_raven_2 = df_raven_2[~df_raven_2["tuid"].isin(bad_tuids)]

In [10]:
group_a = df_raven_2[df_raven_2["ugroup"] == "A"]
group_b = df_raven_2[df_raven_2["ugroup"] == "B"]
group_c = df_raven_2[df_raven_2["ugroup"] == "C"]
group_d = df_raven_2[df_raven_2["ugroup"] == "D"]
group_e = df_raven_2[df_raven_2["ugroup"] == "E"]

In [37]:
# create a syntetic group - F
def is_fully_confident(question_id):
    return question_id in [
        '218', 
        '219',
        '1228_b',
        '1229',
        '6999',
        '7108'
    ]

def extract_question_num(s):
    last_parts = s.split('_')[-2:]
    if last_parts[-1].isdigit():
        result = last_parts[-1]
    else:
        result = "_".join(last_parts)
    return result
    
def create_group_F(users_logs):
    results = {
        "participant_id": [],
        "condition": [],
        "accuracy": []
    }

    users_logs = users_logs[users_logs["ugroup"] == 'E']
    users = users_logs['tuid'].unique()
    
    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        correct_count = 0
        questions = 0
       
        for _, row in user_data.iterrows():
            question = extract_question_num(row['blockId'])
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            pred_choice = convert_2_int(row['predictedShapeChoiceIndex'])

            is_fully_confident_question = is_fully_confident(question)
            user_choice = pred_choice if is_fully_confident_question else convert_2_int(row['selectedShapeIndex'])
              
            if pd.notna(row['stepActionType']):
                questions += 1
                
                # check how many qeusiotns does the user answered correctly
                if user_choice == true_choice:
                    correct_count += 1

        results["participant_id"].append(user)
        results["condition"].append("F")
        results["accuracy"].append(correct_count/questions)

    return pd.DataFrame(results)

In [38]:
def analysis_test_results(users_logs, group_id):
    results = {
        "participant_id": [],
        "condition": [],
        "accuracy": []
    }
    
    users_logs = users_logs[users_logs["ugroup"] == group_id]
    users = users_logs['tuid'].unique()
    
    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        correct_count = 0
        questions = 0

        for _, row in user_data.iterrows():
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            user_choice = convert_2_int(row['selectedShapeIndex'])
            
            if pd.notna(row['stepActionType']):
                questions += 1
                
                # check how many qeusiotns does the user answered correctly
                if user_choice == true_choice:
                    correct_count += 1
        

        results["participant_id"].append(user)
        results["condition"].append(group_id)
        results["accuracy"].append(correct_count/questions)
        

    return pd.DataFrame(results)

In [39]:
group_a_data = analysis_test_results(df_raven_2, "A")
group_b_data = analysis_test_results(df_raven_2, "B")
group_c_data = analysis_test_results(df_raven_2, "C")
group_d_data = analysis_test_results(df_raven_2, "D")
group_e_data = analysis_test_results(df_raven_2, "E")
group_f_data = create_group_F(df_raven_2)

In [40]:
results = pd.concat(
    [group_a_data, group_b_data, group_c_data, group_d_data, group_e_data, group_f_data],
    ignore_index=True
)

In [52]:
def run_group_accuracy_tests(results):
    # -------------------------
    # Kruskal–Wallis
    # -------------------------
    conditions = results["condition"].unique()
    groups = [
        results.loc[results["condition"] == c, "accuracy"].values
        for c in conditions
    ]

    H, p = kruskal(*groups)

    N = len(results)
    k = results["condition"].nunique()
    epsilon_sq = (H - k + 1) / (N - k)

    print(
        f"Kruskal–Wallis: H = {H:.4f}, df = {k-1}, "
        f"p = {p:.6f}, ε² = {epsilon_sq:.4f}"
    )

    # -------------------------
    # Dunn post-hoc (Holm)
    # -------------------------
    posthoc = None
    if p < 0.05:
        posthoc = sp.posthoc_dunn(
            results,
            val_col="accuracy",
            group_col="condition",
            p_adjust="holm"
        )
        print("\nDunn post-hoc (Holm-corrected p-values):")
        print(posthoc)

    # -------------------------
    # Cliff's delta
    # -------------------------
    def cliffs_delta(a, b):
        a = np.asarray(a)
        b = np.asarray(b)
        n_a = len(a)
        n_b = len(b)
        gt = sum(x > y for x in a for y in b)
        lt = sum(x < y for x in a for y in b)
        return (gt - lt) / (n_a * n_b)

    cliffs_results = {}
    conds = list(conditions)

    for i in range(len(conds)):
        for j in range(i + 1, len(conds)):
            c1, c2 = conds[i], conds[j]
            a = results.loc[results["condition"] == c1, "accuracy"].values
            b = results.loc[results["condition"] == c2, "accuracy"].values
            d = cliffs_delta(a, b)
            cliffs_results[(c1, c2)] = d
            print(f"Cliff's δ ({c1} vs {c2}) = {d:.4f}")

    # -------------------------
    # Return structured output
    # -------------------------
    summary = {
        "kruskal": {
            "H": H,
            "df": k - 1,
            "p": p,
            "epsilon_sq": epsilon_sq,
        },
        "posthoc_dunn": posthoc,
        "cliffs_delta": cliffs_results,
    }

    return summary

In [53]:
summary = run_group_accuracy_tests(results)

Kruskal–Wallis: H = 50.5530, df = 5, p = 0.000000, ε² = 0.3996

Dunn post-hoc (Holm-corrected p-values):
              A         B         C         D         E             F
A  1.000000e+00  0.001107  0.000456  0.003182  0.000020  7.000738e-11
B  1.106690e-03  1.000000  1.000000  1.000000  1.000000  2.388973e-02
C  4.562330e-04  1.000000  1.000000  1.000000  1.000000  4.379043e-02
D  3.181575e-03  1.000000  1.000000  1.000000  1.000000  9.997687e-03
E  2.034768e-05  1.000000  1.000000  1.000000  1.000000  2.514533e-01
F  7.000738e-11  0.023890  0.043790  0.009998  0.251453  1.000000e+00
Cliff's δ (A vs B) = -0.7425
Cliff's δ (A vs C) = -0.8325
Cliff's δ (A vs D) = -0.7950
Cliff's δ (A vs E) = -0.8675
Cliff's δ (A vs F) = -0.9650
Cliff's δ (B vs C) = -0.0300
Cliff's δ (B vs D) = 0.0575
Cliff's δ (B vs E) = -0.1750
Cliff's δ (B vs F) = -0.5850
Cliff's δ (C vs D) = 0.1375
Cliff's δ (C vs E) = -0.1350
Cliff's δ (C vs F) = -0.6100
Cliff's δ (D vs E) = -0.2475
Cliff's δ (D vs F) = -0.6500
C